# 🧪 Optimized Domain-Specific Fine-Tuning (FIXED VERSION)
### Fine-Tuning LLMs on Your PDF Data (Non-Instruction / Continued Pre-training)

This version fixes the `ValueError` by ensuring `SFTTrainer` manages the LoRA process internally.

In [ ]:
!pip install -U bitsandbytes transformers accelerate peft trl PyMuPDF datasets

In [ ]:
import os
import fitz  # PyMuPDF
import re
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer

print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Advanced PDF Extraction & Cleaning

In [ ]:
def extract_and_clean_pdf(pdf_path):
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"File not found: {pdf_path}")
    
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text_blocks.append(page.get_text())
    
    full_text = " ".join(text_blocks)
    # Join broken lines
    full_text = re.sub(r'(?<![\.?!\:])\n\s*', ' ', full_text)
    full_text = re.sub(r'\s+', ' ', full_text)
    
    chunks = [full_text[i:i+1000] for i in range(0, len(full_text), 1000)] 
    return [{"text": c.strip()} for c in chunks if len(c.strip()) > 50]

PDF_PATH = "Metformin.pdf"
data = extract_and_clean_pdf(PDF_PATH)
dataset = Dataset.from_list(data).train_test_split(test_size=0.1)
print(f"Dataset prepared: {len(dataset['train'])} train samples.")

## 2. Model Loading & LoRA Configuration

In [ ]:
model_id = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Step 1: Prepare for kbit training
model = prepare_model_for_kbit_training(model)

# Step 2: Define PEFT config (but don't wrap the model yet!)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

## 3. Training with SFTTrainer

In [ ]:
training_args = TrainingArguments(
    output_dir="./llama-domain-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4, 
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,        # Pass base model
    peft_config=peft_config, # Pass config
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_len=512,
    packing=True,
    args=training_args,
)

trainer.train()

## 4. Test Generation

In [ ]:
prompt = "Metformin is a medication commonly prescribed to "
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = trainer.model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))